# Homework 8 — solution by (your_name)

Read the homework PDF first: it has the rules. The questions are quoted in this notebook, in blockquotes (with a bar on
their left), so that your answers stand apart. Copy this notebook to
`3dasm_course/Assignments/your_Assignments/HW8/your_Homework8.ipynb` and work there.

**How this notebook works** (as in the lectures and in Homeworks 4 to 6):

1. the cell `%%writefile CLAUDE.md` writes the instructions file that the agent of this homework reads: **you** write it;
2. the prompt is a `PROMPT_...` cell, run with `run_agent`: **you** write it. The agent works in its own task folder
   inside `HW8/` (`HW8/ex2_2/`);
3. after the prompt, a cell that we provide loads the agent's `results.json`. The keys it needs are written just above
   the prompt cell: your prompt must make the agent store exactly those. If one is missing, the cell tells you which.

This is a **short** homework (two exercises), so that you have time for the Final Project.

**Your prompt is graded** (see the PDF): write it in your own words, and make it define the problem uniquely.
A prompt that only points to a question ("solve exercise 2.2"), or pastes it, counts as not answered.

In [ ]:
# Provided by the course: the helper that runs a coding agent from this notebook (see agent_helper.py)
import sys; sys.path.insert(0, "../../..")          # the folder 3dasm_course, where agent_helper.py is
from agent_helper import set_model, run_agent, results, figures, show, save_results, task_folder
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

set_model("sonnet")        # "sonnet" or "haiku": the TA runs your notebook again with this same model
import sklearn; print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)   # some results depend on them

## Your instructions file for the agent

Write here everything that does not depend on the task: the environment, the packages, where files go, and what the
agent must report. The lectures and Homeworks 4 to 6 have examples.

In [ ]:
%%writefile CLAUDE.md
# YOUR CLAUDE.md HERE

## Exercise 1 (pen and paper)

Binary logistic regression models a class $y \in \{0, 1\}$ with the observation distribution $p(y|\mathbf{x}, \mathbf{w}) = \text{Ber}(y|\rho)$, $\rho = \sigma(a)$, $a = \mathbf{w}^T\boldsymbol{\phi}(\mathbf{x})$, where $\sigma(a) = 1/(1+e^{-a})$ is the sigmoid.

> **1.1.** **The sigmoid**: show that $\sigma(-a) = 1 - \sigma(a)$ and that $\frac{d\sigma}{da} = \sigma(a)\left(1-\sigma(a)\right)$. Linear regression used the linear function $\mathbf{w}^T\boldsymbol{\phi}(\mathbf{x})$ directly as the mean of a Gaussian: why can it not be used directly as the parameter $\rho$ of the Bernoulli distribution?

*YOUR ANSWER HERE*

> **1.2.** **The decision boundary**: let $\boldsymbol{\phi}(\mathbf{x}) = [1, x_1, x_2]^T$. Show that the predicted class (the mode of the PPD) is $\hat{y}^* = 1$ if and only if $w_0 + w_1x_1^* + w_2x_2^* > 0$, so that the decision boundary is a straight line. On which line is $\rho = 0.9$, and on which is $\rho = 0.1$? Show that the distance between these two lines is $2\ln 9/\sqrt{w_1^2+w_2^2}$, and say what happens to it when all the weights are multiplied by 10.

*YOUR ANSWER HERE*

> **1.3.** **Separable classes**: suppose that some weights $\mathbf{w}$ classify every training point correctly, with $\mathbf{w}^T\boldsymbol{\phi}(\mathbf{x}_n) > 0$ when $y_n = 1$ and $< 0$ when $y_n = 0$. Show that the negative log likelihood of $t\mathbf{w}$ decreases as $t > 1$ grows, and tends to 0. What does this say about the maximum likelihood estimate of $\mathbf{w}$ for separable classes? Using 1.2, what does the PPD look like for very large $t$?

*YOUR ANSWER HERE*

## Exercise 2 (with the agent)

In this exercise you measure what the hyperparameter `C` of scikit-learn’s [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) does, on two problems built from the Iris dataset (`data_iris.csv`, in the folder `Assignments/data`), each one with two inputs and the bias:

-   **Problem A**: all 150 flowers; inputs petal length and petal width; class 1 is *Iris-setosa*, class 0 the other two species.

-   **Problem B**: only the 100 flowers of *Iris-versicolor* and *Iris-virginica*; inputs sepal length and sepal width; class 1 is *Iris-virginica*.

Use all the flowers of each problem for training (there is no testing set: this exercise is about the fit, not about the predictions), the raw inputs (no scaling), the `lbfgs` solver with `tol=1e-10` and `max_iter=100000`, and $\texttt{C} \in \{10^{-2}, 1, 10^{2}, 10^{4}, 10^{6}\}$.

> **2.1.** **Before running anything**, look at the flowers of each problem (the notebook plots them for you) and predict: in which problem will the weights $(w_1, w_2)$ keep growing as `C` grows, and why? (You will not lose points for wrong predictions. You will lose points for not writing them.)

In [ ]:
# Provided: the flowers of the two problems
iris = pd.read_csv("../../data/data_iris.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for s, color, label in [("Iris-setosa", "purple", "class 1: setosa"), ("other", "slategray", "class 0: versicolor and virginica")]:
    d = iris[iris.Species == s] if s != "other" else iris[iris.Species != "Iris-setosa"]
    ax[0].scatter(d.PetalLengthCm, d.PetalWidthCm, c=color, edgecolor="black", s=20, label=label)
for s, color, label in [("Iris-virginica", "saddlebrown", "class 1: virginica"), ("Iris-versicolor", "teal", "class 0: versicolor")]:
    d = iris[iris.Species == s]
    ax[1].scatter(d.SepalLengthCm, d.SepalWidthCm, c=color, edgecolor="black", s=20, label=label)
ax[0].set(title="Problem A", xlabel="petal length (cm)", ylabel="petal width (cm)")
ax[1].set(title="Problem B", xlabel="sepal length (cm)", ylabel="sepal width (cm)")
for a in ax: a.legend(fontsize=9)
plt.show()

*YOUR ANSWER HERE*

> **2.2.** Write a prompt that fits the ten models and stores, for each problem and each `C`: the weights $w_1, w_2$, the bias $w_0$, $\|(w_1, w_2)\|$, the negative log likelihood of the training data (the sum over the flowers, with the natural logarithm), and the training accuracy. Show, for each problem, the flowers with the decision boundary and the lines $\rho = 0.1$ and $\rho = 0.9$ for `C`$=1$ and `C`$=10^6$.

Your prompt must make the agent store in `results.json`: `A` and `B`, each a dictionary with the lists `C`, `w0`, `w1`, `w2`, `norm_w`, `nll` and `accuracy` (5 values each, in the order $\texttt{C} = 10^{-2}, 1, 10^{2}, 10^{4}, 10^{6}$), where `norm_w` is $\|(w_1, w_2)\|$.

In [ ]:
TASK_2_2 = "ex2_2"          # the task name (the cells below use it)
PROMPT_2_2 = """YOUR PROMPT HERE"""
run_agent(TASK_2_2, PROMPT_2_2)

In [ ]:
# Provided: the results of task ex2_2
r = results(TASK_2_2, ["A", "B"])
for p in ("A", "B"):
    print(f"Problem {p}")
    print(pd.DataFrame({k: r[p][k] for k in ("C", "w0", "w1", "w2", "norm_w", "nll", "accuracy")}).to_string(index=False))
show(*figures(TASK_2_2), width=900)

*YOUR ANSWER HERE*

> **2.3.** **Check** (your own code, in a notebook cell, using only the stored weights and the data): compute the gradient of the negative log likelihood, $\boldsymbol{\Phi}^T(\boldsymbol{\rho}-\mathbf{y})$, at each of the ten fits. It is not zero: compare it, component by component, with $-\mathbf{w}/\texttt{C}$, where the bias component of $\mathbf{w}$ is set to zero (differences of about $10^{-6}$ or smaller are the precision of the solver, not a disagreement). Which objective does scikit-learn minimize? Write it, and identify `C` in terms of a Gaussian prior on the weights. Why is the bias treated differently?

In [ ]:
# YOUR CHECK HERE (your own code: the stored weights and the data only)

*YOUR ANSWER HERE*

> **2.4.** **What can you conclude?** Compare with your predictions of 2.1 and with Exercise 1 (1.2 and 1.3): why do the weights converge in one problem and not in the other, and how does the width of the band between $\rho = 0.1$ and $\rho = 0.9$ change? In Problem A, compute from the stored weights the probability that a flower with petal length 2.8 cm and petal width 0.9 cm is setosa, for `C`$=1$ and `C`$=10^6$. This flower lies in the gap between setosa and the other species: would you trust either probability? What does the prior buy you, and what would you need to choose `C`?

*YOUR ANSWER HERE*